In [12]:
# ============================================================
# File Name: BBIM502_Assessment2.py
# Student Name: Gosai Krishpuri Hiteshpuri
# Student ID: 850007190
# Project: Langham Hotel Management System
# Description: Console-based hotel room management system with
#              file handling and exception handling (11 built-in
#              exceptions handled).
# Date: 4 November 2026
# ============================================================

# ------------------------------------------------------------
# ImportError handling: stop cleanly if a required module is
# missing instead of crashing with a long error message.
# ------------------------------------------------------------
try:
    import os                      # file existence check
    import ast                     # safe parsing of typed numbers
    import math                    # isfinite() check for numbers
    from datetime import datetime  # timestamp for backup file name
except ImportError as error:
    print(f"A required module could not be imported: {error}")
    raise SystemExit(1)

# Student ID used to create the allocation file
STUDENT_ID = "850007190"
ALLOCATION_FILE = f"LHMS_{STUDENT_ID}.txt"

# GST rate (15%)
GST_RATE = 0.15

# Room types selected by number (1 = Standard, 2 = Deluxe, 3 = Suite)
ROOM_TYPES = ["Standard", "Deluxe", "Suite"]

# Maximum nights allowed for one booking
MAX_NIGHTS = 365

# Store room details and allocations
rooms = {}
allocations = {}


# ------------------------------------------------------------
# Helper: read a number safely
# Handles: SyntaxError, ValueError, TypeError, OverflowError
# Returns the number, or None if the entry was invalid.
# ------------------------------------------------------------
def read_number(prompt, whole_number=False):
    """
     Read a number typed by the user.
     Returns the number, or None if the entry is invalid.
    """
    text = input(prompt).strip()

    try:
        # literal_eval turns the typed text into a Python value safely
        value = ast.literal_eval(text)

        # TypeError: the value is not a number (e.g. text or True)
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("a numeric value is required")

        # ValueError: a whole number was needed but a decimal was given
        if whole_number and not isinstance(value, int):
            raise ValueError("a whole number is required")

        # OverflowError: float() fails if an integer is far too large
        number = float(value)

        # ValueError: reject infinity (e.g. 1e999)
        if not math.isfinite(number):
            raise ValueError("number must be finite")

        return value if whole_number else number

    except SyntaxError:
        print("Invalid format. Type digits only, for example 150 or 150.50.")
    except ValueError:
        print("Invalid value. Please enter a valid number "
              "(a whole number where required).")
    except TypeError:
        print("Wrong data type. A numeric value is required.")
    except OverflowError:
        print("That number is too large to process.")

    return None


# ------------------------------------------------------------
# Helper: choose a room type from the ROOM_TYPES list
# Handles: IndexError (number outside the list)
# ------------------------------------------------------------
def choose_room_type():
    print("Room types: 1. Standard  2. Deluxe  3. Suite")
    type_number = read_number("Enter room type (1/2/3): ", whole_number=True)

    if type_number is None:
        return None

    try:
        # Negative or zero positions would silently pick the wrong type
        if type_number < 1:
            raise IndexError("room type must be 1 or higher")

        return ROOM_TYPES[type_number - 1]

    except IndexError:
        print("Invalid room type. Please choose 1, 2 or 3.")
        return None


# ------------------------------------------------------------
# Helper: calculate subtotal, GST and total for a booking
# Handles: TypeError, OverflowError, NameError
# ------------------------------------------------------------
def calculate_bill(price, nights):
    try:
        subtotal = round(price * nights, 2)
        gst_amount = round(subtotal * GST_RATE, 2)
        total_bill = round(subtotal + gst_amount, 2)

        # Python floats overflow to infinity instead of raising an error
        if not math.isfinite(total_bill):
            raise OverflowError("bill exceeds the maximum numeric limit")

        return subtotal, gst_amount, total_bill

    except TypeError:
        print("Billing error: price and nights must both be numbers.")
    except OverflowError:
        print("Billing error: the bill is too large to calculate.")
    except NameError as error:
        print(f"Billing error: a required value is not defined ({error}).")

    return None


# ------------------------------------------------------------
# Helper: average cost per night
# Handles: ZeroDivisionError (zero nights)
# ------------------------------------------------------------
def calculate_average_per_night(total_bill, nights):
    try:
        return total_bill / nights
    except ZeroDivisionError:
        print("Cannot calculate average per night: nights is zero.")
        return 0.0


# ------------------------------------------------------------
# Helper: percentage of rooms currently allocated
# Handles: ZeroDivisionError (no rooms added yet)
# ------------------------------------------------------------
def calculate_occupancy_rate():
    allocated_count = sum(1 for details in rooms.values() if details["allocated"])

    try:
        return (allocated_count / len(rooms)) * 100
    except ZeroDivisionError:
        return 0.0


# ------------------------------------------------------------
# Option 1: Add Room
# ------------------------------------------------------------
def add_room():
    print("\n--- Add Room ---")

    room_number = input("Enter room number: ").strip()

    if not room_number:
        print("Room number cannot be empty.")
        return

    if room_number in rooms:
        print("This room already exists.")
        return

    room_type = choose_room_type()
    if room_type is None:
        return

    room_price = read_number("Enter price per night: ")
    if room_price is None:
        return

    if room_price <= 0:
        print("Price must be greater than zero.")
        return

    rooms[room_number] = {
        "type": room_type,
        "price": room_price,
        "allocated": False
    }

    print("Room added successfully!")


# ------------------------------------------------------------
# Option 2: Delete Room
# ------------------------------------------------------------
def delete_room():
    print("\n--- Delete Room ---")

    room_number = input("Enter room number to delete: ").strip()

    if room_number not in rooms:
        print("Room not found.")
        return

    if rooms[room_number]["allocated"]:
        print("Cannot delete an allocated room.")
        return

    del rooms[room_number]

    print("Room deleted successfully!")


# ------------------------------------------------------------
# Option 3: Display Room Details
# ------------------------------------------------------------
def display_rooms():
    print("\n--- Room Details ---")

    if not rooms:
        print("No rooms have been added.")
    else:
        print(f"{'Room':<12}{'Type':<15}{'Price/Night':<15}{'Status'}")
        print("-" * 55)

        for room_number, details in rooms.items():
            status = "Allocated" if details["allocated"] else "Available"

            print(
                f"{room_number:<12}"
                f"{details['type']:<15}"
                f"${details['price']:<14.2f}"
                f"{status}"
            )

    print(f"\nOccupancy rate: {calculate_occupancy_rate():.1f}%")


# ------------------------------------------------------------
# Option 4: Allocate Room
# ------------------------------------------------------------
def allocate_room():
    print("\n--- Allocate Room ---")

    room_number = input("Enter room number: ").strip()

    if room_number not in rooms:
        print("Room not found. Please add the room first.")
        return

    if rooms[room_number]["allocated"]:
        print("This room is already allocated.")
        return

    customer_name = input("Enter customer name: ").strip()

    if not customer_name:
        print("Customer name cannot be empty.")
        return

    customer_id = input("Enter customer ID: ").strip()

    if not customer_id:
        print("Customer ID cannot be empty.")
        return

    nights = read_number("Enter number of nights: ", whole_number=True)
    if nights is None:
        return

    if nights <= 0 or nights > MAX_NIGHTS:
        print(f"Number of nights must be between 1 and {MAX_NIGHTS}.")
        return

    allocations[room_number] = {
        "customer_name": customer_name,
        "customer_id": customer_id,
        "nights": nights,
        "room_type": rooms[room_number]["type"],
        "price": rooms[room_number]["price"]
    }

    rooms[room_number]["allocated"] = True

    print("Room allocated successfully!")


# ------------------------------------------------------------
# Option 5: Display Room Allocation Details
# ------------------------------------------------------------
def display_allocations():
    print("\n--- Room Allocation Details ---")

    if not allocations:
        print("There are no room allocations.")
        return

    for room_number, details in allocations.items():

        bill = calculate_bill(details["price"], details["nights"])
        if bill is None:
            continue

        subtotal, gst_amount, total_bill = bill

        print("-" * 40)
        print(f"Room Number: {room_number}")
        print(f"Room Type: {details['room_type']}")
        print(f"Customer Name: {details['customer_name']}")
        print(f"Customer ID: {details['customer_id']}")
        print(f"Number of Nights: {details['nights']}")
        print(f"Price per Night: ${details['price']:.2f}")
        print(f"Subtotal: ${subtotal:.2f}")
        print(f"GST (15%): ${gst_amount:.2f}")
        print(f"Estimated Bill Including GST: ${total_bill:.2f}")


# ------------------------------------------------------------
# Option 6: Billing and De-allocation
# ------------------------------------------------------------
def billing_and_deallocation():
    print("\n--- Billing and De-allocation ---")

    room_number = input("Enter room number: ").strip()

    if room_number not in allocations:
        print("No allocation found for this room.")
        return

    details = allocations[room_number]

    bill = calculate_bill(details["price"], details["nights"])
    if bill is None:
        return

    subtotal, gst_amount, total_bill = bill
    average_per_night = calculate_average_per_night(total_bill, details["nights"])

    print("\n--- Customer Bill ---")
    print(f"Room Number: {room_number}")
    print(f"Customer Name: {details['customer_name']}")
    print(f"Customer ID: {details['customer_id']}")
    print(f"Number of Nights: {details['nights']}")
    print(f"Price per Night: ${details['price']:.2f}")
    print(f"Subtotal: ${subtotal:.2f}")
    print(f"GST (15%): ${gst_amount:.2f}")
    print(f"Total Bill Including GST: ${total_bill:.2f}")
    print(f"Average Cost per Night (incl. GST): ${average_per_night:.2f}")

    confirm = input(
        "Confirm payment and de-allocate this room? (yes/no): "
    ).strip().lower()

    if confirm == "yes":
        del allocations[room_number]
        rooms[room_number]["allocated"] = False

        print("Payment recorded and room de-allocated.")

    elif confirm == "no":
        print("Billing completed without de-allocation.")

    else:
        print("Answer not recognised. Room was not de-allocated.")


# ------------------------------------------------------------
# Option 7: Save Room Allocations to File
# Handles: FileNotFoundError, IOError
# ------------------------------------------------------------
def save_allocations():
    print("\n--- Save Room Allocations to File ---")

    try:
        # "w" creates the file if it does not exist; "with" closes it
        # automatically when the block ends
        with open(ALLOCATION_FILE, "w", encoding="utf-8") as file:

            if not allocations:
                file.write("No room allocations currently exist.\n")

            for room_number, details in allocations.items():

                bill = calculate_bill(details["price"], details["nights"])

                file.write(f"Room Number: {room_number}\n")
                file.write(f"Customer Name: {details['customer_name']}\n")
                file.write(f"Customer ID: {details['customer_id']}\n")
                file.write(f"Room Type: {details['room_type']}\n")
                file.write(f"Nights: {details['nights']}\n")
                file.write(f"Price per Night: ${details['price']:.2f}\n")

                if bill is None:
                    file.write("Bill could not be calculated.\n")
                else:
                    subtotal, gst_amount, total = bill
                    file.write(f"Subtotal: ${subtotal:.2f}\n")
                    file.write(f"GST (15%): ${gst_amount:.2f}\n")
                    file.write(f"Total Bill Including GST: ${total:.2f}\n")

                file.write("-" * 35 + "\n")

        print(f"Allocations saved to {ALLOCATION_FILE}")

    except FileNotFoundError:
        print("File error: the folder for the allocation file was not found.")

    except IOError as error:
        print(f"File error while saving allocations: {error}")


# ------------------------------------------------------------
# Option 8: Display Room Allocations from File
# Handles: FileNotFoundError, IOError
# ------------------------------------------------------------
def display_allocations_from_file():
    print("\n--- Display Allocations from File ---")

    try:
        with open(ALLOCATION_FILE, "r", encoding="utf-8") as file:
            contents = file.read()

        if contents.strip():
            print(contents)
        else:
            print("The allocation file is empty.")

    except FileNotFoundError:
        print("Allocation file not found. Please save allocations first.")

    except IOError as error:
        print(f"File error while reading allocations: {error}")


# ------------------------------------------------------------
# Option 9: Back Up and Clear Room Allocation File
# Handles: FileNotFoundError, IOError
# ------------------------------------------------------------
def backup_and_clear_file():
    print("\n--- Back Up and Clear Allocation File ---")

    if not os.path.exists(ALLOCATION_FILE):
        print("Allocation file does not exist. Save allocations first.")
        return

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_file = f"LHMS_{STUDENT_ID}_Backup_{timestamp}.txt"

    try:
        # Step 1: read the original file
        with open(ALLOCATION_FILE, "r", encoding="utf-8") as source:
            contents = source.read()

        # Do not create an empty backup
        if not contents.strip():
            print("The allocation file is already empty. Nothing to back up.")
            return

        # Step 2: append the content to the backup file
        with open(backup_file, "a", encoding="utf-8") as backup:
            backup.write(contents)

        # Step 3: clear the original file (opening in "w" mode empties it)
        with open(ALLOCATION_FILE, "w", encoding="utf-8") as source:
            source.write("")

        print(f"Backup created: {backup_file}")
        print(f"Original file cleared: {ALLOCATION_FILE}")

    except FileNotFoundError:
        print("File error: the allocation file could not be found.")

    except IOError as error:
        print(f"File error during backup: {error}")


# ------------------------------------------------------------
# Display Main Menu
# ------------------------------------------------------------
def display_menu():
    print("\n==========================================")
    print("     LANGHAM HOTEL MANAGEMENT SYSTEM")
    print("==========================================")
    print("1. Add Room")
    print("2. Delete Room")
    print("3. Display Room Details")
    print("4. Allocate Rooms")
    print("5. Display Room Allocation Details")
    print("6. Billing and De-Allocation")
    print("7. Save Room Allocation to File")
    print("8. Display Room Allocation from File")
    print("9. Back Up and Clear Room Allocation File")
    print("0. Exit Application")
    print("==========================================")


# ------------------------------------------------------------
# Main Program
# Handles: EOFError, NameError and a safety net for any error
# that was not already handled inside the menu functions.
# ------------------------------------------------------------
def main():
    while True:
        display_menu()

        try:
            choice = input("Enter your choice (0-9): ").strip()

            if choice == "1":
                add_room()

            elif choice == "2":
                delete_room()

            elif choice == "3":
                display_rooms()

            elif choice == "4":
                allocate_room()

            elif choice == "5":
                display_allocations()

            elif choice == "6":
                billing_and_deallocation()

            elif choice == "7":
                save_allocations()

            elif choice == "8":
                display_allocations_from_file()

            elif choice == "9":
                backup_and_clear_file()

            elif choice == "0":
                print("Thank you for using Langham Hotel Management System.")
                break

            else:
                print("Invalid choice. Enter a number from 0 to 9.")

        except EOFError:
            # Input stream ended (e.g. Ctrl+Z / Ctrl+D)
            print("\nInput ended. Exiting application.")
            break

        except (KeyboardInterrupt, SystemExit):
            print("\nApplication stopped.")
            break

        except NameError as error:
            print(f"Program error: a name is not defined ({error}).")

        except FileNotFoundError:
            print("File error: the requested file was not found.")

        except IOError as error:
            print(f"File error: {error}")

        except (ValueError, TypeError, ZeroDivisionError,
                IndexError, OverflowError) as error:
            print(f"Input or calculation error: {error}")

        except Exception as error:
            print(f"An unexpected error occurred: {error}")

In [9]:
main()


     LANGHAM HOTEL MANAGEMENT SYSTEM
1. Add Room
2. Delete Room
3. Display Room Details
4. Allocate Rooms
5. Display Room Allocation Details
6. Billing and De-Allocation
7. Save Room Allocation to File
8. Display Room Allocation from File
9. Back Up and Clear Room Allocation File
0. Exit Application


Enter your choice (0-9):  1



--- Add Room ---


Enter room number:  1104


Room types: 1. Standard  2. Deluxe  3. Suite


Enter room type (1/2/3):  1
Enter price per night:  10000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000


... (7 outputs left out)

Enter your choice (0-9):  8



--- Display Allocations from File ---
The allocation file is empty.

     LANGHAM HOTEL MANAGEMENT SYSTEM
1. Add Room
2. Delete Room
3. Display Room Details
4. Allocate Rooms
5. Display Room Allocation Details
6. Billing and De-Allocation
7. Save Room Allocation to File
8. Display Room Allocation from File
9. Back Up and Clear Room Allocation File
0. Exit Application


Enter your choice (0-9):  0


Thank you for using Langham Hotel Management System.


In [8]:
print("1" + "0" * 400)

10000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000


In [10]:
# ============================================================
# File Name: BBIM502_Exception_Demo.py
# Student Name: Gosai Krishpuri Hiteshpuri
# Student ID: 850007190
# Description: Deliberately triggers each of the 11 required
#              built-in exceptions and shows it being handled
#              with try / except (evidence for Task 2).
# Date: 4 November 2026
# ============================================================

# 1. SyntaxError - code text with a syntax mistake
try:
    compile("room_price = = 150", "<demo>", "exec")
except SyntaxError as error:
    print(f"1. SyntaxError handled: {error.msg}")

# 2. ValueError - right type, inappropriate value
try:
    nights = int("three")
except ValueError as error:
    print(f"2. ValueError handled: {error}")

# 3. ZeroDivisionError - dividing by zero nights
try:
    average = 575.00 / 0
except ZeroDivisionError as error:
    print(f"3. ZeroDivisionError handled: {error}")

# 4. IndexError - position outside the list
try:
    room_types = ["Standard", "Deluxe", "Suite"]
    chosen = room_types[5]
except IndexError as error:
    print(f"4. IndexError handled: {error}")

# 5. NameError - variable that was never defined
try:
    print(total_bill_amount)
except NameError as error:
    print(f"5. NameError handled: {error}")

# 6. TypeError - operator used with the wrong data type
try:
    result = 150.00 + "2 nights"
except TypeError as error:
    print(f"6. TypeError handled: {error}")

# 7. OverflowError - number too large for the numeric type
try:
    huge = float(10 ** 400)
except OverflowError as error:
    print(f"7. OverflowError handled: {error}")

# 8. IOError - a folder cannot be opened as a file
try:
    with open(".", "r") as file:
        file.read()
except IOError as error:
    print(f"8. IOError handled: {error}")

# 9. ImportError - module that does not exist
try:
    import langham_payment_module
except ImportError as error:
    print(f"9. ImportError handled: {error}")

# 10. EOFError - end of input reached without reading any data
#     (raised manually here because Jupyter's input() box never ends)
try:
    raise EOFError("EOF when reading a line")
except EOFError as error:
    print(f"10. EOFError handled: {error}")

# 11. FileNotFoundError - file name that does not exist
try:
    with open("LHMS_wrong_filename.txt", "r") as file:
        file.read()
except FileNotFoundError as error:
    print(f"11. FileNotFoundError handled: {error}")

print("\nAll 11 exceptions were triggered and handled without crashing.")

1. SyntaxError handled: invalid syntax
2. ValueError handled: invalid literal for int() with base 10: 'three'
3. ZeroDivisionError handled: division by zero
4. IndexError handled: list index out of range
5. NameError handled: name 'total_bill_amount' is not defined
6. TypeError handled: unsupported operand type(s) for +: 'float' and 'str'
7. OverflowError handled: int too large to convert to float
8. IOError handled: [Errno 13] Permission denied: '.'
9. ImportError handled: No module named 'langham_payment_module'
10. EOFError handled: EOF when reading a line
11. FileNotFoundError handled: [Errno 2] No such file or directory: 'LHMS_wrong_filename.txt'

All 11 exceptions were triggered and handled without crashing.
